# Laboratorium 2: Współbieżność i Równoległość w Pythonie

### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---


## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---


### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).**

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`.

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:

1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).


In [ ]:
import requests
from bs4 import BeautifulSoup
import time


def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")
    event_titles = [item.text.strip() for item in soup.select(".item__link h3")]
    return event_titles


def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]

    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()

    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))

    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")

    print(f"\nCzas wykonania: {time.time() - start:.2f}s")


run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Szalone nożyczki (Teatr Bagatela)
2. Atrament dla leworęcznych. Komedia absurdalna
3. Karty na stół
4. Nasze żony
5. Dizajn i emocje – jak działa na nas projektowanie?
6. Martwe natury
7. Bajki dla niegrzecznych
8. Hexvessel, Aluk Todolo, BaarRa w Zaścianku
9. Eros Ramazzotti: Una storia importante w Tauron Arenie Kraków
10. Wróg ludu

Czas wykonania: 2.55s


In [ ]:
import concurrent.futures


def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]

    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))

    all_titles = [title for sublist in results for title in sublist]

    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")

    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")


run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Szalone nożyczki (Teatr Bagatela)
2. Atrament dla leworęcznych. Komedia absurdalna
3. Karty na stół
4. Nasze żony
5. Dizajn i emocje – jak działa na nas projektowanie?
6. Martwe natury
7. Bajki dla niegrzecznych
8. Hexvessel, Aluk Todolo, BaarRa w Zaścianku
9. Eros Ramazzotti: Una storia importante w Tauron Arenie Kraków
10. Wróg ludu

Czas wykonania (wątki): 0.72s


---

## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).


In [ ]:
import threading


class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001)  # Symulacja opóźnienia
            self.balance = current + amount


account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))

print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


---

## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.


In [1]:
import multiprocessing
import time

# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes


def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()

    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)

    print(f"Zakończono w czasie {time.time() - start:.2f}s.")


if __name__ == "__main__":
    run_primes_demo()

Praca na 8 procesach (rdzeniach)...


Zakończono w czasie 1.13s.


---

# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.


### Zadanie 1 (Threading)

Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:

1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

_Podpowiedź: Użyj `requests.get(URL).json().get('fact')`_


In [ ]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"
NUM_FACTS = 20


def fetch_cat_fact(i):
    try:
        response = requests.get(CAT_API_URL, timeout=5)
        response.raise_for_status()
        return response.json().get("fact")
    except Exception as e:
        print(f"Błąd pobierania faktu {i+1}: {e}")
        return None


def get_facts_sequential(num_facts):
    facts = []
    start = time.time()
    for i in range(num_facts):
        fact = fetch_cat_fact(i)
        if fact:
            facts.append(fact)
    duration = time.time() - start
    return facts, duration


def get_facts_threaded(num_facts, max_workers=5):
    start = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = executor.map(fetch_cat_fact, range(num_facts))
    facts = [fact for fact in results if fact]
    duration = time.time() - start
    return facts, duration


# Sekwencyjne pobieranie
seq_facts, seq_time = get_facts_sequential(NUM_FACTS)
print(f"\n--- Sekwencyjne pobieranie ({seq_time:.2f} sekund) ---")
for i, fact in enumerate(seq_facts, 1):
    print(f"{i}. {fact}")

# Wielowątkowe pobieranie
thread_facts, thread_time = get_facts_threaded(NUM_FACTS)
print(f"\n--- Wielowątkowe pobieranie ({thread_time:.2f} sekund) ---")
for i, fact in enumerate(thread_facts, 1):
    print(f"{i}. {fact}")

# Porównanie
percent_of_seq = (thread_time / seq_time) * 100 if seq_time > 0 else 0

print(f"\nSekwencyjny wykonał się w {seq_time:.2f} s,")
print(
    f"wielowątkowy w {thread_time:.2f} s, co stanowi ~{percent_of_seq:.0f}% czasu sekwencyjnego."
)


--- Sekwencyjne pobieranie (4.63 sekund) ---
1. Researchers believe the word “tabby” comes from Attabiyah, a neighborhood in Baghdad, Iraq. Tabbies got their name because their striped coats resembled the famous wavy patterns in the silk produced in this city.
2. A female cat is called a queen or a molly.
3. Cats are now Britain's favourite pet: there are 7.7 million cats as opposed to 6.6 million dogs.
4. A healthy cat has a temperature between 38 and 39 degrees Celcius.
5. When a cats rubs up against you, the cat is marking you with it's scent claiming ownership.
6. 70% of your cat's life is spent asleep.
7. If they have ample water, cats can tolerate temperatures up to 133 °F.
8. Tigers are excellent swimmers and do not avoid water.
9. Cat families usually play best in even numbers. Cats and kittens should be aquired in pairs whenever possible.
10. The first true cats came into existence about 12 million years ago and were the Proailurus.
11. According to Hebrew legend, Noah prayed

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)

Napisz program o strukturze **producent-consumers**:

1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.


In [ ]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

NUM_ITEMS = 20

q = queue.Queue()

producer_done = threading.Event()


def producer():
    for i in range(1, NUM_ITEMS + 1):
        q.put(i)
        print(f"Producent dodał: {i}")
        time.sleep(0.1)
    producer_done.set()


def consumer_even():
    while not (producer_done.is_set() and q.empty()):
        try:
            item = q.get(timeout=0.2)
            if item % 2 == 0:
                print(f"Konsument 1 (parzyste) pobrał: {item}")
            else:
                q.put(item)
            q.task_done()
        except queue.Empty:
            continue


def consumer_odd():
    while not (producer_done.is_set() and q.empty()):
        try:
            item = q.get(timeout=0.2)
            if item % 2 != 0:
                print(f"Konsument 2 (nieparzyste) pobrał: {item}")
            else:
                q.put(item)
            q.task_done()
        except queue.Empty:
            continue


t_producer = threading.Thread(target=producer)
t_consumer_even = threading.Thread(target=consumer_even)
t_consumer_odd = threading.Thread(target=consumer_odd)

t_producer.start()
t_consumer_even.start()
t_consumer_odd.start()

t_producer.join()
t_consumer_even.join()
t_consumer_odd.join()

print("Przetwarzanie zakończone.")

Producent dodał: 1
Konsument 2 (nieparzyste) pobrał: 1
Producent dodał: 2
Konsument 1 (parzyste) pobrał: 2
Producent dodał: 3
Konsument 2 (nieparzyste) pobrał: 3
Producent dodał: 4
Konsument 1 (parzyste) pobrał: 4
Producent dodał: 5
Konsument 2 (nieparzyste) pobrał: 5
Producent dodał: 6
Konsument 1 (parzyste) pobrał: 6
Producent dodał: 7
Konsument 2 (nieparzyste) pobrał: 7
Producent dodał: 8
Konsument 1 (parzyste) pobrał: 8
Producent dodał: 9
Konsument 2 (nieparzyste) pobrał: 9
Producent dodał: 10
Konsument 1 (parzyste) pobrał: 10
Producent dodał: 11
Konsument 2 (nieparzyste) pobrał: 11
Producent dodał: 12
Konsument 1 (parzyste) pobrał: 12
Producent dodał: 13
Konsument 2 (nieparzyste) pobrał: 13
Producent dodał: 14
Konsument 1 (parzyste) pobrał: 14
Producent dodał: 15
Konsument 2 (nieparzyste) pobrał: 15
Producent dodał: 16
Konsument 1 (parzyste) pobrał: 16
Producent dodał: 17
Konsument 2 (nieparzyste) pobrał: 17
Producent dodał: 18
Konsument 1 (parzyste) pobrał: 18
Producent dodał: 19

### Zadanie 3 (Multiprocessing)

Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).


In [2]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    start = 1
    end = 10000
    numbers = list(range(start, end + 1))

    with multiprocessing.Pool() as pool:
        start_time = time.time()
        results = pool.map(calculate_power_sum, numbers)
        end_time = time.time()

    print(f"Obliczono sumy potęg dla liczb {start}-{end}")
    print(f"Czas wykonania: {end_time - start_time:.2f} sekund")
    print(f"Przykład wyników (dla liczb 1-10): ")
    for n, res in zip(numbers[:10], results[:10]):
        print(f"{n} -> {res}")

Obliczono sumy potęg dla liczb 1-10000
Czas wykonania: 0.18 sekund
Przykład wyników (dla liczb 1-10): 
1 -> 100
2 -> 2535301200456458802993406410750
3 -> 773066281098016996554691694648431909053161283000
4 -> 2142584059011987034055949456454883470029603991710390447068500
5 -> 9860761315262647567646607066034827870915080438862787559628486633300780
6 -> 783982348200085087316028320589669384644572452567545845851686359643396569772850
7 -> 3773555927895550989902089063950252946000070398722062967756211219956973369576416070000
8 -> 2328041115810841241449652215325003612630249592761070000727017656405007199729527664209597000
9 -> 298815737485359115506128987290252080182887634235068807971396831956479052263964955868682786424500
10 -> 11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111110
